In [ ]:
import numpy as np
from sklearn.metrics import f1_score

def pick_threshold_on_val(y_val, proba_val, grid=np.linspace(0.05, 0.95, 181)):
    y_val = np.asarray(y_val)
    best_t, best_f1 = 0.5, -1.0
    for t in grid:
        f1 = f1_score(y_val, (proba_val >= t).astype(int), pos_label=1, zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, float(t)
    return best_t, best_f1

In [ ]:
import pandas as pd

BASE = '/home/jupyter/workspace/rw-migration-aou-rw-24b38658'
MATRIX_DIR = f'{BASE}/amia/new_survey'

matrices = {}
for k in ['ehrdemo_6', 'ehrdemo_survnobasic_6',
          'ehrdemo_12', 'ehrdemo_survnobasic_12',
          'ehrdemo_24', 'ehrdemo_survnobasic_24']:
    matrices[k] = pd.read_parquet(f'{MATRIX_DIR}/matrix_{k}.parquet')
    print(f"read {k}: {matrices[k].shape}")

In [ ]:
import pyarrow.parquet as pq

def load_lean(path):
    pf = pq.ParquetFile(path)
    schema = pf.schema_arrow
    names = schema.names

    out = {}
    for name in names:
        col = pq.read_table(path, columns=[name]).column(0).to_pandas()
        if name == 'person_id':
            out[name] = col.astype('int64')
        elif name == 'IsPositive':
            out[name] = col.astype('int8')
        elif col.dtype == 'float64':
            out[name] = col.astype('float32')
        elif col.dtype == 'int64':
            out[name] = col.astype('int32')
        else:
            out[name] = col
        del col

    df = pd.DataFrame(out)
    del out
    gc.collect()
    return df

In [ ]:
import numpy as np
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             precision_score, recall_score, f1_score)

def evaluate(y_true, proba, threshold=None):
    y_true = np.asarray(y_true)
    if threshold is None:
        threshold, _ = pick_threshold_on_val(y_true, proba)   
    y_pred = (proba >= threshold).astype(int)
    return {
        'Macro_AUC':  roc_auc_score(y_true, proba),
        'Precision+': precision_score(y_true, y_pred, pos_label=1, zero_division=0),
        'Recall+':    recall_score(y_true, y_pred, pos_label=1, zero_division=0),
        'F1+':        f1_score(y_true, y_pred, pos_label=1, zero_division=0),
        'Macro_F1':   f1_score(y_true, y_pred, average='macro'),
        'ROC_AUC':    roc_auc_score(y_true, proba),
        'PR_AUC':     average_precision_score(y_true, proba),
        'threshold':  threshold,
    }

In [ ]:
import pandas as pd
BASE='/home/jupyter/workspace/rw-migration-aou-rw-24b38658'
MATRIX_DIR=f'{BASE}/amia/matrix'
df = pd.read_parquet(f'{MATRIX_DIR}/matrix_ehrdemo_12.parquet')
feat=[c for c in df.columns if c not in ('person_id','IsPositive')]
pos=df[df['IsPositive']==1]; neg=df[df['IsPositive']==0]
rows=[]
for c in feat:
    pz=(pos[c].fillna(0)==0).mean(); nz=(neg[c].fillna(0)==0).mean()
    rows.append((c, round(pz,3), round(nz,3), round(abs(pz-nz),3)))
scan=pd.DataFrame(rows,columns=['feat','pos_zero','neg_zero','gap']).sort_values('gap',ascending=False)
print("gap max 15:")
print(scan.head(15).to_string(index=False))
print("\nneg_zero>=0.99:", (scan['neg_zero']>=0.99).sum())
print("pos_zero>=0.99:", (scan['pos_zero']>=0.99).sum())

In [ ]:
import numpy as np, gc, os, joblib, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split

RANDOM_STATE, TEST_SIZE = 42, 0.20
BASE='/home/jupyter/workspace/rw-migration-aou-rw-24b38658'
MODEL_DIR=f'{BASE}/amia/new_survey/models'; SCORE_DIR=f'{BASE}/amia/new_survey/risk_scores'
os.makedirs(MODEL_DIR, exist_ok=True); os.makedirs(SCORE_DIR, exist_ok=True)

def run_lr(matrices, evaluate, timeframes=(6,12,24), versions=('ehrdemo','ehrdemo_survnobasic'),
           C=1.0, verbose=True, save=True):
    results, preds = [], {}
    for tf in timeframes:
        for ver in versions:
            label = f'{ver}_{tf}m'
            df = matrices[f'{ver}_{tf}']
            y   = df['IsPositive'].to_numpy()
            pid = df['person_id'].to_numpy()
            feat_cols = [c for c in df.columns if c not in ('person_id', 'IsPositive')]
            
            X = df[feat_cols].to_numpy(dtype='float32', copy=False)
            del df, matrices[f'{ver}_{tf}']; gc.collect()

            idx = np.arange(len(y))
            idx_tr, idx_te = train_test_split(idx, test_size=TEST_SIZE, stratify=y,
                                              random_state=RANDOM_STATE)
            idx_tr, idx_va = train_test_split(idx_tr, test_size=0.15, stratify=y[idx_tr],
                                              random_state=RANDOM_STATE)

            med = np.nanmedian(X[idx_tr], axis=0).astype('float32')
            med = np.where(np.isnan(med), np.float32(0), med)   
            def prep(rows, mu=None, sd=None):
                A = X[rows].copy()                             
                nan_mask = np.isnan(A)
                A[nan_mask] = np.take(med, np.where(nan_mask)[1])
                if mu is None:
                    mu = A.mean(axis=0, dtype='float32')
                    sd = A.std(axis=0, dtype='float32')
                    sd[sd == 0] = np.float32(1)
                A -= mu; A /= sd                              
                return A, mu, sd

            X_tr, mu, sd = prep(idx_tr)
            y_tr = y[idx_tr]
            lr = LogisticRegression(penalty='l2', C=C, max_iter=1000, random_state=RANDOM_STATE)
            lr.fit(X_tr, y_tr)
            del X_tr; gc.collect()

            X_va, _, _ = prep(idx_va, mu, sd)
            va_p = lr.predict_proba(X_va)[:, 1]
            del X_va; gc.collect()

            X_te, _, _ = prep(idx_te, mu, sd)
            te_p = lr.predict_proba(X_te)[:, 1]
            del X_te, X; gc.collect()

            y_va, y_te = y[idx_va], y[idx_te]
            thr, thr_f1 = pick_threshold_on_val(y_va, va_p)
            m = {'model': 'LogisticReg', 'matrix': label, **evaluate(y_te, te_p, threshold=thr)}
            results.append(m)

            score_df = pd.concat([
                pd.DataFrame({'person_id': pid[idx_va], 'y_true': y_va, 'proba': va_p, 'split': 'val'}),
                pd.DataFrame({'person_id': pid[idx_te], 'y_true': y_te, 'proba': te_p, 'split': 'test'}),
            ], ignore_index=True)
            score_df.insert(0, 'model', 'LogisticReg'); score_df.insert(1, 'matrix', label)
            score_df['threshold'] = thr
            preds[label] = score_df

            if save:
                joblib.dump({'model': lr, 'median': med, 'mean': mu, 'std': sd,
                             'feature_cols': feat_cols, 'threshold': thr, 'label': label, 'C': C},
                            f'{MODEL_DIR}/LR_{label}.joblib')
                score_df.to_parquet(f'{SCORE_DIR}/LR_{label}.parquet', index=False)

            if verbose:
                print(f"[LR  {label:<20}] PR-AUC={m['PR_AUC']:.4f} ROC-AUC={m['ROC_AUC']:.4f} "
                      f"F1+={m['F1+']:.4f} (thr={thr:.3f}, val_F1={thr_f1:.4f})")
            del lr; gc.collect()
    return results, preds

In [ ]:
import pandas as pd, gc
MATRIX_DIR=f'{BASE}/amia/new_survey'

lr_results, lr_preds = [], {}
for tf in [6,12,24]:
    for ver in ['ehrdemo','ehrdemo_survnobasic']:
        mats = {f'{ver}_{tf}': load_lean(f'{MATRIX_DIR}/matrix_{ver}_{tf}.parquet')}
        res, pr = run_lr(mats, evaluate, timeframes=(tf,), versions=(ver,))  
        lr_results += res; lr_preds.update(pr)
        del mats; gc.collect()

print(pd.DataFrame(lr_results)[['matrix','ROC_AUC','PR_AUC','F1+']])

In [ ]:
import numpy as np, gc, os, joblib, pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from joblib.externals.loky import get_reusable_executor

RANDOM_STATE, TEST_SIZE = 42, 0.20
BASE='/home/jupyter/workspace/rw-migration-aou-rw-24b38658'
MODEL_DIR=f'{BASE}/amia/new_survey/models'; SCORE_DIR=f'{BASE}/amia/new_survey/risk_scores'
os.makedirs(MODEL_DIR, exist_ok=True); os.makedirs(SCORE_DIR, exist_ok=True)

def run_rf(matrices, evaluate, timeframes=(6,12,24), versions=('ehrdemo','ehrdemo_survnobasic'),
           n_estimators=400, max_depth=None, min_samples_leaf=5, n_jobs=2, verbose=True, save=True):
    results, preds = [], {}
    for tf in timeframes:
        for ver in versions:
            label = f'{ver}_{tf}m'
            df = matrices[f'{ver}_{tf}']
            y   = df['IsPositive']
            pid = df['person_id']                               
            X   = df.drop(columns=['person_id','IsPositive'])
            feat_cols = X.columns.tolist()                     

            X_tr, X_te, y_tr, y_te, pid_tr, pid_te = train_test_split(   
            X, y, pid, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE)
            X_tr, X_va, y_tr, y_va, pid_tr, pid_va = train_test_split(
                X_tr, y_tr, pid_tr, test_size=0.15, stratify=y_tr, random_state=RANDOM_STATE)
            del X, df; gc.collect()

            imp = SimpleImputer(strategy='median')             
            X_tr_f = imp.fit_transform(X_tr).astype('float32')
            X_va_f = imp.transform(X_va).astype('float32')
            X_te_f = imp.transform(X_te).astype('float32')
            del X_tr, X_va, X_te; gc.collect()

            rf = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth,
                                        min_samples_leaf=min_samples_leaf,
                                        n_jobs=n_jobs, random_state=RANDOM_STATE)
            rf.fit(X_tr_f, y_tr)
            va_p = rf.predict_proba(X_va_f)[:,1]
            te_p = rf.predict_proba(X_te_f)[:,1]
            thr, thr_f1 = pick_threshold_on_val(np.asarray(y_va), va_p)
            m = {'model':'RandomForest','matrix':label, **evaluate(y_te, te_p, threshold=thr)}
            results.append(m)
            
            score_df = pd.concat([
                pd.DataFrame({'person_id':np.asarray(pid_va),'y_true':np.asarray(y_va),'proba':va_p,'split':'val'}),
                pd.DataFrame({'person_id':np.asarray(pid_te),'y_true':np.asarray(y_te),'proba':te_p,'split':'test'}),
            ], ignore_index=True)
            score_df.insert(0,'model','RandomForest'); score_df.insert(1,'matrix',label); score_df['threshold']=thr
            preds[label] = score_df

            if save:                                       
                joblib.dump({'model':rf,'imputer':imp,'scaler':None,'feature_cols':feat_cols,
                             'threshold':thr,'label':label,
                             'params':{'n_estimators':n_estimators,'max_depth':max_depth,'min_samples_leaf':min_samples_leaf}},
                            f'{MODEL_DIR}/RF_{label}.joblib')
                score_df.to_parquet(f'{SCORE_DIR}/RF_{label}.parquet', index=False)

            if verbose:
                print(f"[RF  {label:<20}] PR-AUC={m['PR_AUC']:.4f} ROC-AUC={m['ROC_AUC']:.4f} "
                      f"F1+={m['F1+']:.4f} (thr={thr:.3f}, val_F1={thr_f1:.4f})")
            del rf, X_tr_f, X_va_f, X_te_f, imp; gc.collect()
            get_reusable_executor().shutdown(wait=True)      
            gc.collect()
    return results, preds

In [ ]:
rf_results, rf_preds = [], {}
for tf in [6,12,24]:
    for ver in ['ehrdemo','ehrdemo_survnobasic']:
        mats = {f'{ver}_{tf}': load_lean(f'{MATRIX_DIR}/matrix_{ver}_{tf}.parquet')}
        res, pr = run_rf(mats, evaluate, timeframes=(tf,), versions=(ver,))
        rf_results += res; rf_preds.update(pr); del mats; gc.collect()
print(pd.DataFrame(rf_results)[['matrix','ROC_AUC','PR_AUC','F1+']])

In [ ]:
pip install xgboost

In [ ]:
import xgboost as xgb
import numpy as np, gc, os, joblib, pandas as pd
from sklearn.model_selection import train_test_split

RANDOM_STATE, TEST_SIZE = 42, 0.20
BASE='/home/jupyter/workspace/rw-migration-aou-rw-24b38658'
MODEL_DIR=f'{BASE}/amia/new_survey/models'; SCORE_DIR=f'{BASE}/amia/new_survey/risk_scores'
os.makedirs(MODEL_DIR, exist_ok=True); os.makedirs(SCORE_DIR, exist_ok=True)

XGB_PARAMS = dict(
    objective="binary:logistic",
    eval_metric=["auc", "aucpr"],   
    eta=0.03, max_depth=5, min_child_weight=16,
    subsample=0.75, colsample_bytree=0.70,
    reg_lambda=3.0, reg_alpha=0.4, tree_method="hist",
)

def run_xgb(matrices, evaluate, timeframes=(6,12,24), versions=('ehrdemo','ehrdemo_survnobasic'),
            num_boost_round=5000, early_stopping_rounds=200, verbose=True, save=True):
    results, preds = [], {}
    for tf in timeframes:
        for ver in versions:
            label = f'{ver}_{tf}m'
            df = matrices[f'{ver}_{tf}']
            y   = df['IsPositive']
            pid = df['person_id']                               
            X   = df.drop(columns=['person_id','IsPositive'])
            feat_cols = X.columns.tolist()                      

            X_tr, X_te, y_tr, y_te, pid_tr, pid_te = train_test_split(   
                X, y, pid, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE)
            X_tr, X_va, y_tr, y_va, pid_tr, pid_va = train_test_split(
                X_tr, y_tr, pid_tr, test_size=0.15, stratify=y_tr, random_state=RANDOM_STATE)
            del X, df; gc.collect()

            dtr = xgb.DMatrix(X_tr, label=y_tr)                 
            dva = xgb.DMatrix(X_va, label=y_va)
            dte = xgb.DMatrix(X_te, label=y_te)
            del X_tr, X_va, X_te; gc.collect()

            bst = xgb.train(XGB_PARAMS, dtr, num_boost_round=num_boost_round,
                            evals=[(dva, "valid")], early_stopping_rounds=early_stopping_rounds,
                            verbose_eval=False)
            it = (0, bst.best_iteration + 1)
            va_p = bst.predict(dva, iteration_range=it)
            te_p = bst.predict(dte, iteration_range=it)
            thr, thr_f1 = pick_threshold_on_val(np.asarray(y_va), va_p)
            m = {'model':'XGBoost','matrix':label, **evaluate(y_te, te_p, threshold=thr)}
            results.append(m)

            score_df = pd.concat([
                pd.DataFrame({'person_id':np.asarray(pid_va),'y_true':np.asarray(y_va),'proba':va_p,'split':'val'}),
                pd.DataFrame({'person_id':np.asarray(pid_te),'y_true':np.asarray(y_te),'proba':te_p,'split':'test'}),
            ], ignore_index=True)
            score_df.insert(0,'model','XGBoost'); score_df.insert(1,'matrix',label); score_df['threshold']=thr
            preds[label] = score_df

            if save:                                          
                joblib.dump({'model':bst,'imputer':None,'scaler':None,'feature_cols':feat_cols,
                             'best_iteration':int(bst.best_iteration),'threshold':thr,'label':label,
                             'params':XGB_PARAMS},
                            f'{MODEL_DIR}/XGB_{label}.joblib')
                score_df.to_parquet(f'{SCORE_DIR}/XGB_{label}.parquet', index=False)

            if verbose:
                print(f"[XGB {label:<20}] PR-AUC={m['PR_AUC']:.4f} ROC-AUC={m['ROC_AUC']:.4f} "
                      f"F1+={m['F1+']:.4f} (best_iter={bst.best_iteration}, thr={thr:.3f}, val_F1={thr_f1:.4f})")
            del bst, dtr, dva, dte; gc.collect()
    return results, preds

In [ ]:
import pandas as pd, gc

xgb_results, xgb_preds = [], {}
for tf in [6, 12, 24]:
    for ver in ['ehrdemo', 'ehrdemo_survnobasic']:
        mats = {f'{ver}_{tf}': load_lean(f'{MATRIX_DIR}/matrix_{ver}_{tf}.parquet')}
        res, pr = run_xgb(mats, evaluate, timeframes=(tf,), versions=(ver,))
        xgb_results += res
        xgb_preds.update(pr)
        del mats; gc.collect()

print(pd.DataFrame(xgb_results)[['matrix','ROC_AUC','PR_AUC','F1+','Precision+','Recall+','Macro_F1']].to_string(index=False))

In [ ]:
import numpy as np, gc, os, joblib, pandas as pd
import torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score

RANDOM_STATE, TEST_SIZE = 42, 0.20
BASE='/home/jupyter/workspace/rw-migration-aou-rw-24b38658'
MODEL_DIR=f'{BASE}/amia/new_survey/models'; SCORE_DIR=f'{BASE}/amia/new_survey/risk_scores'
os.makedirs(MODEL_DIR, exist_ok=True); os.makedirs(SCORE_DIR, exist_ok=True)

class MLP(nn.Module):
    def __init__(self, in_dim, hidden=(256,128), p_drop=0.3):
        super().__init__()
        layers, prev = [], in_dim
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(p_drop)]
            prev = h
        layers += [nn.Linear(prev, 1)]
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x).squeeze(1)

def run_mlp(matrices, evaluate, timeframes=(6,12,24), versions=('ehrdemo','ehrdemo_survnobasic'),
            hidden=(256,128), p_drop=0.3, max_epochs=60, patience=8,
            batch_size=2048, lr=1e-3, weight_decay=1e-5, verbose=True, save=True):
    torch.set_num_threads(4)
    results, preds = [], {}
    for tf in timeframes:
        for ver in versions:
            label = f'{ver}_{tf}m'
            df = matrices[f'{ver}_{tf}']
            y   = df['IsPositive'].values.astype('float32')
            pid = df['person_id'].values                        
            X   = df.drop(columns=['person_id','IsPositive'])
            feat_cols = X.columns.tolist()                      

            Xtr, Xte, ytr, yte, pidtr, pidte = train_test_split(   
                X, y, pid, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE)
            del X, df; gc.collect()
            Xtr, Xval, ytr, yval, pidtr, pidval = train_test_split(
                Xtr, ytr, pidtr, test_size=0.15, stratify=ytr, random_state=RANDOM_STATE)

            imp = SimpleImputer(strategy='median')
            Xtr  = imp.fit_transform(Xtr).astype('float32')
            Xval = imp.transform(Xval).astype('float32')
            Xte  = imp.transform(Xte).astype('float32')
            scaler = StandardScaler(copy=False)
            Xtr  = scaler.fit_transform(Xtr)
            Xval = scaler.transform(Xval); Xte = scaler.transform(Xte)
            in_dim = Xtr.shape[1]

            tr_dl = DataLoader(TensorDataset(torch.from_numpy(Xtr), torch.from_numpy(ytr)),
                               batch_size=batch_size, shuffle=True, drop_last=True)
            Xval_t, Xte_t = torch.from_numpy(Xval), torch.from_numpy(Xte)
            del Xtr, Xval, Xte; gc.collect()

            torch.manual_seed(RANDOM_STATE)
            model = MLP(in_dim, hidden, p_drop)
            opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
            loss_fn = nn.BCEWithLogitsLoss()
            best_ap, best_state, wait = -1.0, None, 0
            for epoch in range(max_epochs):
                model.train()
                for xb, yb in tr_dl:
                    opt.zero_grad(); loss = loss_fn(model(xb), yb); loss.backward(); opt.step()
                model.eval()
                with torch.no_grad():
                    val_proba = torch.sigmoid(model(Xval_t)).numpy()
                ap = average_precision_score(yval, val_proba)
                if ap > best_ap:
                    best_ap = ap
                    best_state = {k: v.clone() for k, v in model.state_dict().items()}
                    wait = 0
                else:
                    wait += 1
                    if wait >= patience: break
            model.load_state_dict(best_state); model.eval()
            with torch.no_grad():
                val_proba = torch.sigmoid(model(Xval_t)).numpy()
                proba     = torch.sigmoid(model(Xte_t)).numpy()
            thr, thr_f1 = pick_threshold_on_val(yval, val_proba)
            m = {'model':'MLP','matrix':label, **evaluate(yte, proba, threshold=thr)}
            results.append(m)
            
            score_df = pd.concat([
                pd.DataFrame({'person_id':np.asarray(pidval),'y_true':yval,'proba':val_proba,'split':'val'}),
                pd.DataFrame({'person_id':np.asarray(pidte), 'y_true':yte, 'proba':proba,    'split':'test'}),
            ], ignore_index=True)
            score_df.insert(0,'model','MLP'); score_df.insert(1,'matrix',label); score_df['threshold']=thr
            preds[label] = score_df

            if save:                                            
                joblib.dump({'state_dict':best_state,'in_dim':in_dim,'hidden':hidden,'p_drop':p_drop,
                             'imputer':imp,'scaler':scaler,'feature_cols':feat_cols,
                             'threshold':thr,'label':label,'model':'MLP'},
                            f'{MODEL_DIR}/MLP_{label}.joblib')
                score_df.to_parquet(f'{SCORE_DIR}/MLP_{label}.parquet', index=False)

            if verbose:
                print(f"[MLP {label:<20}] PR-AUC={m['PR_AUC']:.4f} ROC-AUC={m['ROC_AUC']:.4f} "
                      f"F1+={m['F1+']:.4f} (epochs={epoch+1}, val_AP={best_ap:.4f}, thr={thr:.3f})")
            del model, tr_dl, Xval_t, Xte_t, best_state; gc.collect()
    return results, preds

In [ ]:
mlp_results, mlp_preds = [], {}
for tf in [6,12,24]:
    for ver in ['ehrdemo','ehrdemo_survnobasic']:
        mats = {f'{ver}_{tf}': load_lean(f'{MATRIX_DIR}/matrix_{ver}_{tf}.parquet')}
        res, pr = run_mlp(mats, evaluate, timeframes=(tf,), versions=(ver,))
        mlp_results += res; mlp_preds.update(pr); del mats; gc.collect()
print(pd.DataFrame(mlp_results)[['matrix','ROC_AUC','PR_AUC','F1+','Precision+','Recall+']].to_string(index=False))

In [ ]:
LightGBM

In [ ]:
pip install lightgbm

In [ ]:
import numpy as np, gc, os, joblib, pandas as pd
import lightgbm as lgb
from sklearn.model_selection import train_test_split

RANDOM_STATE, TEST_SIZE = 42, 0.20
BASE='/home/jupyter/workspace/rw-migration-aou-rw-24b38658'
MODEL_DIR=f'{BASE}/amia/new_survey/models'; SCORE_DIR=f'{BASE}/amia/new_survey/risk_scores'
os.makedirs(MODEL_DIR, exist_ok=True); os.makedirs(SCORE_DIR, exist_ok=True)

LGB_PARAMS = dict(
    objective="binary", metric=["auc", "average_precision"],
    learning_rate=0.03, num_leaves=31, max_depth=-1,
    min_child_samples=40, subsample=0.75, subsample_freq=1,
    colsample_bytree=0.70, reg_lambda=3.0, reg_alpha=0.4,
    n_jobs=4, verbosity=-1,
)

def run_lgb(matrices, evaluate, timeframes=(6,12,24), versions=('ehrdemo','ehrdemo_survnobasic'),
            num_boost_round=5000, early_stopping_rounds=200, verbose=True, save=True):
    results, preds = [], {}
    for tf in timeframes:
        for ver in versions:
            label = f'{ver}_{tf}m'
            df = matrices[f'{ver}_{tf}']
            y   = df['IsPositive']
            pid = df['person_id']                               
            X   = df.drop(columns=['person_id','IsPositive'])
            feat_cols = X.columns.tolist()                      

            X_tr, X_te, y_tr, y_te, pid_tr, pid_te = train_test_split(   
                X, y, pid, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE)
            X_tr, X_va, y_tr, y_va, pid_tr, pid_va = train_test_split(
                X_tr, y_tr, pid_tr, test_size=0.15, stratify=y_tr, random_state=RANDOM_STATE)
            del X, df; gc.collect()

            dtr = lgb.Dataset(X_tr, label=y_tr)                 
            dva = lgb.Dataset(X_va, label=y_va, reference=dtr)
            bst = lgb.train(LGB_PARAMS, dtr, num_boost_round=num_boost_round,
                            valid_sets=[dva], valid_names=['valid'],
                            callbacks=[lgb.early_stopping(early_stopping_rounds, verbose=False),
                                       lgb.log_evaluation(0)])
            va_p = bst.predict(X_va, num_iteration=bst.best_iteration)
            te_p = bst.predict(X_te, num_iteration=bst.best_iteration)
            thr, thr_f1 = pick_threshold_on_val(np.asarray(y_va), va_p)
            m = {'model':'LightGBM','matrix':label, **evaluate(y_te, te_p, threshold=thr)}
            results.append(m)

            score_df = pd.concat([
                pd.DataFrame({'person_id':np.asarray(pid_va),'y_true':np.asarray(y_va),'proba':va_p,'split':'val'}),
                pd.DataFrame({'person_id':np.asarray(pid_te),'y_true':np.asarray(y_te),'proba':te_p,'split':'test'}),
            ], ignore_index=True)
            score_df.insert(0,'model','LightGBM'); score_df.insert(1,'matrix',label); score_df['threshold']=thr
            preds[label] = score_df

            if save:                                            
                joblib.dump({'model':bst,'imputer':None,'scaler':None,'feature_cols':feat_cols,
                             'best_iteration':int(bst.best_iteration),'threshold':thr,'label':label,
                             'params':LGB_PARAMS},
                            f'{MODEL_DIR}/LGB_{label}.joblib')
                score_df.to_parquet(f'{SCORE_DIR}/LGB_{label}.parquet', index=False)

            if verbose:
                print(f"[LGB {label:<20}] PR-AUC={m['PR_AUC']:.4f} ROC-AUC={m['ROC_AUC']:.4f} "
                      f"F1+={m['F1+']:.4f} (best_iter={bst.best_iteration}, thr={thr:.3f}, val_F1={thr_f1:.4f})")
            del bst, dtr, dva, X_tr, X_va, X_te; gc.collect()
    return results, preds

In [ ]:
lgb_results, lgb_preds = [], {}
for tf in [6,12,24]:
    for ver in ['ehrdemo','ehrdemo_survnobasic']:
        mats = {f'{ver}_{tf}': load_lean(f'{MATRIX_DIR}/matrix_{ver}_{tf}.parquet')}
        res, pr = run_lgb(mats, evaluate, timeframes=(tf,), versions=(ver,))
        lgb_results += res; lgb_preds.update(pr); del mats; gc.collect()
print(pd.DataFrame(lgb_results)[['matrix','ROC_AUC','PR_AUC','F1+','Precision+','Recall+']].to_string(index=False))

In [ ]:
Increment

In [ ]:
for name in dir():
    v = eval(name)
    if name.endswith('results') and isinstance(v, list) and v and isinstance(v[0], dict):
        models = sorted({d.get('model','?') for d in v})
        print(f"{name:20s} -> {len(v)} rows, model: {models}")

In [ ]:
import pandas as pd, numpy as np

_all = []
for name in ['lr_results','rf_results','xgb_results','mlp_results','lgb_results']:
    if name in dir() and isinstance(eval(name), list):
        _all.extend(eval(name))
res = pd.DataFrame(_all)

def split_label(s):
    tf = s.rsplit('_', 1)[1].replace('m','')         
    ver = s.rsplit('_', 1)[0]                         
    return ver, int(tf)
res[['version','tf']] = res['matrix'].apply(lambda s: pd.Series(split_label(s)))

model_order = ['LogisticReg','RandomForest','XGBoost','MLP','LightGBM']
res['model'] = pd.Categorical(res['model'], categories=model_order, ordered=True)

metrics = ['PR_AUC','ROC_AUC','F1+','Precision+','Recall+','Macro_F1']
metrics = [m for m in metrics if m in res.columns]
full = (res.sort_values(['model','tf','version'])
           [['model','tf','version'] + metrics]
           .reset_index(drop=True))
print("="*70, "\ntest standard\n", "="*70)
print(full.to_string(index=False, float_format=lambda x: f'{x:.4f}'))

base_ver = 'ehrdemo'
surv_ver = 'ehrdemo_survnobasic'
rows = []
inc_metrics = [m for m in ['PR_AUC','ROC_AUC','F1+'] if m in res.columns]
for (mdl, tf), g in res.groupby(['model','tf'], observed=True):
    b = g[g['version']==base_ver]
    s = g[g['version']==surv_ver]
    if len(b) and len(s):
        row = {'model':mdl, 'tf':tf}
        for mt in inc_metrics:
            row[f'Δ{mt}'] = float(s[mt].values[0]) - float(b[mt].values[0])
        rows.append(row)
inc = pd.DataFrame(rows).sort_values(['model','tf'])
print("\n", "="*70, "\nSurvey increment(survnobasic − ehrdemo)\n", "="*70)
print(inc.to_string(index=False, float_format=lambda x: f'{x:+.4f}'))

pivot = inc.pivot(index='model', columns='tf', values='ΔPR_AUC')
pivot.columns = [f'{c}m' for c in pivot.columns]
print("\n", "="*70, "\nΔPR-AUC table\n", "="*70)
print(pivot.to_string(float_format=lambda x: f'{x:+.4f}'))

Sequantial

LSTM

In [ ]:
import pandas as pd, numpy as np
BASE = '/home/jupyter/workspace/rw-migration-aou-rw-24b38658'
CLEAN_DIR = f'{BASE}/amia/clean_data'
df = pd.read_csv(f'{CLEAN_DIR}/clean_condition_6.csv')
df = df.dropna(subset=['condition_start_datetime'])   


lens = df.groupby('person_id').size()
print("# of people:", len(lens))
print(lens.describe(percentiles=[.5,.9,.95,.99]))
print("max:", lens.max())

In [ ]:
import pandas as pd
import numpy as np
import gc

CHUNK = 300_000

def load_dates_chunked(fn, dtcol, clean_dir, idx):
    path = f"{clean_dir}/{fn}.csv"
    kept = []
    for chunk in pd.read_csv(path, usecols=['person_id', dtcol],
                             chunksize=CHUNK, low_memory=False):
        chunk[dtcol] = pd.to_datetime(chunk[dtcol], utc=True, errors='coerce').dt.tz_localize(None).dt.normalize()
        chunk = chunk.dropna(subset=[dtcol])
        chunk = chunk.join(idx, on='person_id', how='inner')
        m = (chunk[dtcol] >= chunk['timeframe_start']) & (chunk[dtcol] < chunk['index_date'])
        chunk = chunk.loc[m, ['person_id', dtcol]].rename(columns={dtcol: 'date'})
        kept.append(chunk.drop_duplicates())
    out = pd.concat(kept, ignore_index=True).drop_duplicates()
    del kept; gc.collect()
    return out


def analyze_days_distribution(TF, CLEAN_DIR):
    
    print(f"\n{'='*60}\nTF = {TF}m\n{'='*60}")

    pos = pd.read_csv(f"{CLEAN_DIR}/clean_positive_{TF}.csv",
                      usecols=['person_id','index_date','timeframe_start'])
    neg = pd.read_csv(f"{CLEAN_DIR}/clean_negative_anchor_{TF}.csv",
                      usecols=['person_id','index_date','timeframe_start'])
    pos['IsPositive'] = 1; neg['IsPositive'] = 0
    anchors = pd.concat([pos, neg], ignore_index=True)
    for c in ['index_date','timeframe_start']:
        anchors[c] = pd.to_datetime(anchors[c], utc=True).dt.tz_localize(None).dt.normalize()
    idx = anchors.set_index('person_id')[['index_date','timeframe_start']]
    del pos, neg; gc.collect()

    tables = {
        'condition':   ('clean_condition_{tf}',   'clean_negative_condition_{tf}',   'condition_start_datetime'),
        'drug':        ('clean_drug_{tf}',         'clean_negative_drug_{tf}',         'drug_exposure_start_datetime'),
        'lab':         ('clean_lab_{tf}',          'clean_negative_lab_{tf}',          'measurement_datetime'),
        'measurement': ('clean_measurement_{tf}',  'clean_negative_measurement_{tf}',  'measurement_datetime'),
        'observation': ('clean_observation_{tf}',  'clean_negative_observation_{tf}',  'observation_datetime'),
    }

    all_pd = pd.DataFrame(columns=['person_id', 'date'])
    for name, (fn_pos_t, fn_neg_t, dtcol) in tables.items():
        fn_pos = fn_pos_t.format(tf=TF)
        fn_neg = fn_neg_t.format(tf=TF)
        d = pd.concat([
            load_dates_chunked(fn_pos, dtcol, CLEAN_DIR, idx),
            load_dates_chunked(fn_neg, dtcol, CLEAN_DIR, idx),
        ], ignore_index=True).drop_duplicates()
        print(f"{name:12s}: {len(d):,} rows (person,date)")
        all_pd = pd.concat([all_pd, d[['person_id','date']]], ignore_index=True).drop_duplicates()
        del d; gc.collect()
        print(f"sum: {len(all_pd):,}")

    n_days = all_pd.groupby('person_id')['date'].nunique()
    n_days = n_days.reindex(anchors['person_id'].values).fillna(0).astype(int)
    del all_pd; gc.collect()

    print(f"TF={TF}m Distribution of date counts per person")
    print(n_days.describe(percentiles=[.5,.75,.9,.95,.99]))
    print("max:", n_days.max())
    for k in [5,10,15,20,25,30,40,50]:
        print(f"  <= {k:>2} days cover: {(n_days<=k).mean()*100:5.1f}%")

    merged = anchors[['person_id','IsPositive']].merge(
        n_days.rename('n_days').reset_index().rename(columns={'index':'person_id'}),
        on='person_id', how='left')
    print(merged.groupby('IsPositive')['n_days'].describe(percentiles=[.5,.9,.95]))

    return n_days, merged


BASE = '/home/jupyter/workspace/rw-migration-aou-rw-24b38658'
CLEAN_DIR = f'{BASE}/amia/clean_data'

n_days_12, merged_12 = analyze_days_distribution(12, CLEAN_DIR)
gc.collect()

In [ ]:
n_days_24, merged_24 = analyze_days_distribution(24, CLEAN_DIR)
gc.collect()

In [ ]:
n_days_6, merged_6 = analyze_days_distribution(6, CLEAN_DIR)
gc.collect()

In [ ]:
Gird Construction

In [ ]:
import os, gc, json
import numpy as np, pandas as pd

BASE = '/home/jupyter/workspace/rw-migration-aou-rw-24b38658'
CLEAN_DIR   = f'{BASE}/amia/clean_data'
feature_dir = f'{BASE}/amia/feature'
SEQ_DIR     = f'{BASE}/amia/seq_data'
os.makedirs(SEQ_DIR, exist_ok=True)
TF, K = 6, 20

# --- split ---
with open(f"{feature_dir}/split_person_ids_{TF}.json") as f:
    split = json.load(f)
train_ids, val_ids, test_ids = set(split['train']), set(split['val']), set(split['test'])

pos = pd.read_csv(f"{CLEAN_DIR}/clean_positive_{TF}.csv", usecols=['person_id','index_date','timeframe_start'])
neg = pd.read_csv(f"{CLEAN_DIR}/clean_negative_anchor_{TF}.csv", usecols=['person_id','index_date','timeframe_start'])
pos['IsPositive']=1; neg['IsPositive']=0
anchors = pd.concat([pos,neg], ignore_index=True)
assert anchors['person_id'].is_unique
for c in ['index_date','timeframe_start']:
    s = pd.to_datetime(anchors[c], utc=True, errors='coerce').dt.strftime('%Y%m%d')
    anchors[c+'_i'] = pd.to_numeric(s, errors='coerce').astype('int32')
bounds = anchors[['person_id','timeframe_start_i','index_date_i']].rename(
    columns={'timeframe_start_i':'ts','index_date_i':'ix'})

persons = anchors['person_id'].to_numpy()
N = len(persons)
pid_to_row = {int(p): i for i, p in enumerate(persons)}
y = anchors['IsPositive'].to_numpy().astype('int8')
split_arr = np.where(anchors['person_id'].isin(train_ids), 0,
             np.where(anchors['person_id'].isin(val_ids), 1, 2)).astype('int8')  # 0=tr 1=va 2=te


def load_pairs(fn, dtcol, chunksize=1_000_000):
    p = f"{CLEAN_DIR}/{fn}.csv"
    if not os.path.exists(p): print("file not exist:", fn); return pd.DataFrame(columns=['person_id','day'])
    out=[]
    for ch in pd.read_csv(p, usecols=['person_id',dtcol], dtype={dtcol:str}, chunksize=chunksize):
        day = pd.to_numeric(ch[dtcol].str.slice(0,10).str.replace('-','',regex=False), errors='coerce')
        d = pd.DataFrame({'person_id': ch['person_id'].values, 'day': day.values}).dropna(subset=['day'])
        d['day']=d['day'].astype('int32')
        d = d.merge(bounds, on='person_id', how='inner')
        m = (d['day']>=d['ts']) & (d['day']<d['ix'])
        d = d.loc[m, ['person_id','day']].drop_duplicates()
        if len(d): out.append(d)
        del ch, day, d
    gc.collect()
    return pd.concat(out, ignore_index=True).drop_duplicates() if out else pd.DataFrame(columns=['person_id','day'])

tables = {
 'condition':('condition','condition_start_datetime'),
 'drug':('drug','drug_exposure_start_datetime'),
 'lab':('lab','measurement_datetime'),
 'measurement':('measurement','measurement_datetime'),
 'observation':('observation','observation_datetime'),
}

ev_parts=[]
for name,(mod,dtcol) in tables.items():
    for fn in (f"clean_{mod}_{TF}", f"clean_negative_{mod}_{TF}"):
        ev_parts.append(load_pairs(fn, dtcol))
    print("done", name)
ev = pd.concat(ev_parts, ignore_index=True).drop_duplicates()
del ev_parts; gc.collect()
print("total person-day:", f"{len(ev):,}")

ev = ev.sort_values(['person_id','day'])
day_mat = np.zeros((N, K), dtype='int32')
n_real  = np.zeros(N, dtype='int16')
for pid, g in ev.groupby('person_id', sort=False):
    r = pid_to_row.get(int(pid), -1)
    if r < 0: continue
    days = g['day'].to_numpy()
    if len(days) >= K:
        day_mat[r] = days[-K:]; n_real[r] = K
    else:
        m = len(days)
        day_mat[r,:m] = days
        day_mat[r,m:] = days[-1]    
        n_real[r] = m

np.savez(f"{SEQ_DIR}/grid_{TF}.npz",
         person_id=persons.astype('int64'), day_mat=day_mat,
         n_real=n_real, y=y, split=split_arr)


print("\n[grid saving] N=", N)
print("n_real percentile [50,75,90,95,99]:", np.percentile(n_real,[50,75,90,95,99]))
print("n_real==0 number of people:", int((n_real==0).sum()))
for lab in [0,1]:
    s = n_real[(y==lab)&(n_real>=1)]
    print(f"  {'pos' if lab else 'neg'} n_real>=1: n={len(s):,} median={np.median(s):.0f} P90={np.percentile(s,90):.0f} 20={ (s>=20).mean()*100:.1f}%")

In [ ]:
condition

In [ ]:
def load_cond_counts(fn, code2name, chunksize=1_000_000):
    p = f"{CLEAN_DIR}/{fn}.csv"
    if not os.path.exists(p): print("  without:", fn); return pd.DataFrame(columns=['person_id','day','code','cnt'])
    out=[]
    uc=['person_id','standard_concept_code','standard_concept_name','condition_start_datetime']
    for ch in pd.read_csv(p, usecols=uc, dtype={'condition_start_datetime':str}, chunksize=chunksize):
        day = pd.to_numeric(ch['condition_start_datetime'].str.slice(0,10).str.replace('-','',regex=False), errors='coerce')
        d = pd.DataFrame({'person_id':ch['person_id'].values, 'day':day.values,
                          'code':ch['standard_concept_code'].astype(str).values}).dropna(subset=['day'])
        d['day']=d['day'].astype('int32')
        d = d.merge(bounds, on='person_id', how='inner')
        d = d.loc[(d['day']>=d['ts']) & (d['day']<d['ix']), ['person_id','day','code']]
        if len(d):
            nm = ch[['standard_concept_code','standard_concept_name']].dropna().drop_duplicates()
            for c,n in nm.itertuples(index=False): code2name.setdefault(str(c), str(n))
            out.append(d.groupby(['person_id','day','code'], sort=False).size().reset_index(name='cnt'))
        del ch, day, d
    gc.collect()
    return pd.concat(out, ignore_index=True) if out else pd.DataFrame(columns=['person_id','day','code','cnt'])

code2name={}
cond = pd.concat([load_cond_counts(f"clean_condition_{TF}", code2name),
                  load_cond_counts(f"clean_negative_condition_{TF}", code2name)], ignore_index=True)
cond = cond.groupby(['person_id','day','code'], sort=False)['cnt'].sum().reset_index()
print("condition in-window person-day-code:", f"{len(cond):,}")

n_train = len(train_ids); thr = 0.005*n_train
prev = cond[cond['person_id'].isin(train_ids)].groupby('code')['person_id'].nunique()
vocab_codes = sorted(prev[prev>=thr].index.tolist())
code_to_feat = {c:i for i,c in enumerate(vocab_codes)}
D = len(vocab_codes)
print(f"train n={n_train}, thr={thr:.0f}(0.5%) -> condition vocab D={D}")
print(prev.sort_values(ascending=False).head(12).rename(index=lambda c: f"{c} | {code2name.get(c,'?')[:34]}"))

cond_v = cond[cond['code'].isin(set(vocab_codes))].copy()
cond_v['feat'] = cond_v['code'].map(code_to_feat).astype('int32')
row_s = pd.Series(np.arange(N,dtype='int32'), index=persons)
cond_v['row'] = cond_v['person_id'].map(row_s).astype('int32')

long_grid = pd.DataFrame({'row':np.repeat(np.arange(N,dtype='int32'), K),
                          'slot':np.tile(np.arange(K,dtype='int16'), N),
                          'day':day_mat.reshape(-1)})
long_grid = long_grid[long_grid['day']>0]

sparse = long_grid.merge(cond_v[['row','day','feat','cnt']], on=['row','day'], how='inner')
sparse['val'] = np.log1p(sparse['cnt'].astype('float32')).astype('float32')
sparse = sparse[['row','slot','feat','val']]
print("condition spase none-zero:", f"{len(sparse):,}")

np.savez(f"{SEQ_DIR}/seq_condition_{TF}.npz",
         row=sparse['row'].to_numpy('int32'), slot=sparse['slot'].to_numpy('int16'),
         feat=sparse['feat'].to_numpy('int32'), val=sparse['val'].to_numpy('float32'),
         N=np.int64(N), K=np.int64(K), D=np.int64(D))
with open(f"{SEQ_DIR}/vocab_condition_{TF}.json","w") as f:
    json.dump({"codes":vocab_codes, "names":[code2name.get(c,'') for c in vocab_codes]}, f)
print("saved seq_condition_6.npz + vocab_condition_6.json")


def dense_of(r):
    a=np.zeros((K,D),dtype='float32')
    s=sparse[sparse['row']==r]
    a[s['slot'].to_numpy(), s['feat'].to_numpy()] = s['val'].to_numpy()
    return a

rows_with_cond = np.array(sorted(sparse['row'].unique()))
print("\nwith condition record:", len(rows_with_cond))

bad=0; checked=0
for r in rows_with_cond:
    m=int(n_real[r])
    if 1<=m<K:
        dv=dense_of(int(r)); checked+=1
        if not np.allclose(dv[m-1:], dv[m-1]): bad+=1
print(f"copy unchanged values:checked{checked} rows,bad {bad} rows")

cand=[r for r in rows_with_cond if (y[r]==1 and 3<=n_real[r]<=6)]
for r in cand[:2]:
    print(f"\nrow={r} pid={persons[r]} y={y[r]} n_real={n_real[r]}")
    print("  day_mat:", day_mat[r])
    print("  dense.T (D×K):\n", dense_of(int(r)).T)

In [ ]:
import os, gc, json
import numpy as np, pandas as pd

BASE='/home/jupyter/workspace/rw-migration-aou-rw-24b38658'
CLEAN_DIR=f'{BASE}/amia/clean_data'; feature_dir=f'{BASE}/amia/feature'; SEQ_DIR=f'{BASE}/amia/seq_data'
TF, K = 6, 20

g = np.load(f'{SEQ_DIR}/grid_{TF}.npz')
persons, day_mat, n_real, y, split = g['person_id'], g['day_mat'], g['n_real'], g['y'], g['split']
N = len(persons)

with open(f'{feature_dir}/split_person_ids_{TF}.json') as f:
    train_ids = set(json.load(f)['train'])

pos = pd.read_csv(f"{CLEAN_DIR}/clean_positive_{TF}.csv", usecols=['person_id','index_date','timeframe_start'])
neg = pd.read_csv(f"{CLEAN_DIR}/clean_negative_anchor_{TF}.csv", usecols=['person_id','index_date','timeframe_start'])
anchors = pd.concat([pos,neg], ignore_index=True)
for c in ['index_date','timeframe_start']:
    anchors[c+'_i'] = pd.to_numeric(pd.to_datetime(anchors[c],utc=True,errors='coerce').dt.strftime('%Y%m%d'),errors='coerce').astype('int32')
bounds = anchors[['person_id','timeframe_start_i','index_date_i']].rename(columns={'timeframe_start_i':'ts','index_date_i':'ix'})

row_s = pd.Series(np.arange(N,dtype='int32'), index=persons)
long_grid = pd.DataFrame({'row':np.repeat(np.arange(N,dtype='int32'),K),
                          'slot':np.tile(np.arange(K,dtype='int16'),N),
                          'day':day_mat.reshape(-1)})
long_grid = long_grid[long_grid['day']>0]

def build_seq_count(mod, dtcol, chunksize=1_000_000):
    def load_counts(fn):
        p=f"{CLEAN_DIR}/{fn}.csv"
        if not os.path.exists(p): print(" without:",fn); return pd.DataFrame(columns=['person_id','day','code','cnt'])
        out=[]
        for ch in pd.read_csv(p, usecols=['person_id','standard_concept_code',dtcol], dtype={dtcol:str}, chunksize=chunksize):
            day=pd.to_numeric(ch[dtcol].str.slice(0,10).str.replace('-','',regex=False),errors='coerce')
            d=pd.DataFrame({'person_id':ch['person_id'].values,'day':day.values,
                            'code':ch['standard_concept_code'].astype(str).values}).dropna(subset=['day'])
            d['day']=d['day'].astype('int32')
            d=d.merge(bounds,on='person_id',how='inner')
            d=d.loc[(d['day']>=d['ts'])&(d['day']<d['ix']),['person_id','day','code']]
            if len(d): out.append(d.groupby(['person_id','day','code'],sort=False).size().reset_index(name='cnt'))
            del ch,day,d
        gc.collect()
        return pd.concat(out,ignore_index=True) if out else pd.DataFrame(columns=['person_id','day','code','cnt'])
    ev=pd.concat([load_counts(f"clean_{mod}_{TF}"),load_counts(f"clean_negative_{mod}_{TF}")],ignore_index=True)
    ev=ev.groupby(['person_id','day','code'],sort=False)['cnt'].sum().reset_index()
    thr=0.005*len(train_ids)
    prev=ev[ev['person_id'].isin(train_ids)].groupby('code')['person_id'].nunique()
    vocab=sorted(prev[prev>=thr].index.tolist()); c2f={c:i for i,c in enumerate(vocab)}; D=len(vocab)
    ev=ev[ev['code'].isin(set(vocab))].copy()
    ev['feat']=ev['code'].map(c2f).astype('int32'); ev['row']=ev['person_id'].map(row_s).astype('int32')
    spx=long_grid.merge(ev[['row','day','feat','cnt']],on=['row','day'],how='inner')
    spx['val']=np.log1p(spx['cnt'].astype('float32')).astype('float32')
    np.savez(f'{SEQ_DIR}/seq_{mod}_{TF}.npz', row=spx['row'].to_numpy('int32'),slot=spx['slot'].to_numpy('int16'),
             feat=spx['feat'].to_numpy('int32'),val=spx['val'].to_numpy('float32'),N=np.int64(N),K=np.int64(K),D=np.int64(D))
    with open(f'{SEQ_DIR}/vocab_{mod}_{TF}.json','w') as f: json.dump({'codes':vocab},f)
    print(f"{mod:11s}: vocab D={D:>4}  sparse none-zero={len(spx):>9,}  covering{ev['person_id'].nunique():>7,} people")
    del ev,spx; gc.collect()

In [ ]:
build_seq_count('drug', 'drug_exposure_start_datetime')
build_seq_count('observation', 'observation_datetime')

In [ ]:
import json, pandas as pd
codes = json.load(open(f'{SEQ_DIR}/vocab_observation_{TF}.json'))['codes']
nm = (pd.read_csv(f"{CLEAN_DIR}/clean_observation_{TF}.csv",
                  usecols=['standard_concept_code','standard_concept_name'], dtype=str)
      .drop_duplicates('standard_concept_code')
      .set_index('standard_concept_code')['standard_concept_name'].to_dict())
for c in codes:
    print(c, '|', nm.get(str(c), '?'))

In [ ]:
build_seq_count('lab', 'measurement_datetime')          
build_seq_count('measurement', 'measurement_datetime')  

In [ ]:
import numpy as np, pandas as pd, joblib, gc
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

train_mask = (split == 0)   # 0=train 1=val 2=test
def fit_static(df):
    A = df.to_numpy(dtype='float32')
    imp = SimpleImputer(strategy='median')
    imp.fit(A[train_mask])
    A = imp.transform(A).astype('float32')
    sc = StandardScaler()
    sc.fit(A[train_mask])
    A = sc.transform(A).astype('float32')
    return A, imp, sc
def align(df):
    df = df.drop(columns=[c for c in df.columns if c.lower() in ('ispositive','label','is_positive')], errors='ignore')
    return df.set_index('person_id').reindex(persons)

demo  = pd.read_csv(f'{CLEAN_DIR}/demographic.csv')
surv = pd.read_parquet(f'{BASE}/amia/new_survey/survey_features_{TF}.parquet')
labf  = pd.read_parquet(f'{feature_dir}/lab_features_{TF}.parquet')
measf = pd.read_parquet(f'{feature_dir}/measurement_features_{TF}.parquet')
labnum  = labf[['person_id'] + [c for c in labf.columns  if c.startswith(('lab_mean_','lab_last_'))]]
measnum = measf[['person_id'] + [c for c in measf.columns if c.endswith(('_mean','_last'))]]
print("lab value column:", labnum.shape[1]-1, " | meas value column:", measnum.shape[1]-1) 

demo_a, surv_a = align(demo), align(surv)
labn_a, measn_a = align(labnum), align(measnum)
base = pd.concat([demo_a, labn_a, measn_a], axis=1)
full = pd.concat([base, surv_a], axis=1)
print("static dimension: w/o survey =", base.shape[1], " | w/ survey =", full.shape[1])   

Xs_base, imp_b, sc_b = fit_static(base)
Xs_full, imp_f, sc_f = fit_static(full)
np.savez(f'{SEQ_DIR}/static_{TF}.npz', static_nosurv=Xs_base, static_surv=Xs_full)
joblib.dump({'nosurv':{'cols':base.columns.tolist(),'imputer':imp_b,'scaler':sc_b},
             'surv'  :{'cols':full.columns.tolist(),'imputer':imp_f,'scaler':sc_f}},
            f'{SEQ_DIR}/static_prep_{TF}.joblib')
print("saved static:", Xs_base.shape, Xs_full.shape)
del labf, measf; gc.collect()

In [ ]:
import numpy as np, pandas as pd, gc, os, torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import average_precision_score
torch.set_num_threads(4)

BASE='/home/jupyter/workspace/rw-migration-aou-rw-24b38658'
SEQ_DIR=f'{BASE}/amia/seq_data'; MODEL_DIR=f'{BASE}/amia/new_survey/models'; SCORE_DIR=f'{BASE}/amia/new_survey/risk_scores'
os.makedirs(MODEL_DIR,exist_ok=True); os.makedirs(SCORE_DIR,exist_ok=True)
TF, K = 6, 20

g=np.load(f'{SEQ_DIR}/grid_{TF}.npz'); persons=g['person_id']; y=g['y'].astype('float32'); split=g['split']; n_real=g['n_real'].astype('int64'); N=len(persons)

mods=['condition','drug','observation','lab','measurement']
R,Sl,F,V,offset,Ds=[],[],[],[],0,{}
for mod in mods:
    z=np.load(f'{SEQ_DIR}/seq_{mod}_{TF}.npz'); D=int(z['D']); Ds[mod]=D
    R.append(z['row']); Sl.append(z['slot']); F.append(z['feat'].astype('int32')+offset); V.append(z['val']); offset+=D
D_seq=offset
row=np.concatenate(R); slot=np.concatenate(Sl).astype('int16'); feat=np.concatenate(F); val=np.concatenate(V)
o=np.argsort(row,kind='stable'); row,slot,feat,val=row[o],slot[o],feat[o],val[o]
row_ptr=np.searchsorted(row,np.arange(N+1)).astype('int64')
del R,Sl,F,V,o; gc.collect()
print("D_seq =",D_seq," each module:",Ds," none zero:",f"{len(row):,}")

S=np.load(f'{SEQ_DIR}/static_{TF}.npz'); Xs_base=S['static_nosurv']; Xs_full=S['static_surv']

class DS(Dataset):
    def __init__(self,rows,static): self.rows=rows; self.static=static
    def __len__(self): return len(self.rows)
    def __getitem__(self,i):
        r=self.rows[i]; a,b=row_ptr[r],row_ptr[r+1]
        x=np.zeros((K,D_seq),dtype='float32'); x[slot[a:b],feat[a:b]]=val[a:b]
        L=max(int(n_real[r]),1)                          
        return x, self.static[r], y[r], np.int64(L)

class TwoTower(nn.Module):
    def __init__(self,d_seq,d_static,hs=128,ht=64,p=0.3):
        super().__init__()
        self.lstm=nn.LSTM(d_seq,hs,batch_first=True)
        self.stat=nn.Sequential(nn.Linear(d_static,ht),nn.ReLU(),nn.Dropout(p))
        self.head=nn.Sequential(nn.Linear(hs+ht,64),nn.ReLU(),nn.Dropout(p),nn.Linear(64,1))
    def forward(self,xs,xt,lens):
        packed=nn.utils.rnn.pack_padded_sequence(xs,lens.cpu(),batch_first=True,enforce_sorted=False)
        _,(h,_)=self.lstm(packed)                        
        return self.head(torch.cat([h[-1],self.stat(xt)],1)).squeeze(1)

def predict(model,dl):
    model.eval(); ps=[]
    with torch.no_grad():
        for xs,xt,_,ln in dl: ps.append(torch.sigmoid(model(xs,xt,ln)).numpy())
    return np.concatenate(ps)

def run_lstm(use_survey, max_epochs=30, patience=5, bs=512, lr=1e-3, wd=1e-5):
    ver='ehrdemo_survnobasic' if use_survey else 'ehrdemo'; label=f'{ver}_{TF}m'
    static=Xs_full if use_survey else Xs_base; d_static=static.shape[1]
    tr=np.where(split==0)[0]; va=np.where(split==1)[0]; te=np.where(split==2)[0]
    tr_dl=DataLoader(DS(tr,static),batch_size=bs,shuffle=True,num_workers=0)
    va_dl=DataLoader(DS(va,static),batch_size=1024,shuffle=False,num_workers=0)
    te_dl=DataLoader(DS(te,static),batch_size=1024,shuffle=False,num_workers=0)
    torch.manual_seed(42)
    model=TwoTower(D_seq,d_static); opt=torch.optim.Adam(model.parameters(),lr=lr,weight_decay=wd)
    lossf=nn.BCEWithLogitsLoss(); best_ap,best_state,wait=-1,None,0
    for ep in range(max_epochs):
        model.train()
        for xs,xt,yb,ln in tr_dl:
            opt.zero_grad(); lossf(model(xs,xt,ln),yb).backward(); opt.step()
        ap=average_precision_score(y[va],predict(model,va_dl))
        print(f"  [{label}] epoch {ep+1:>2} val_AP={ap:.4f}")
        if ap>best_ap: best_ap,best_state,wait=ap,{k:v.clone() for k,v in model.state_dict().items()},0
        else:
            wait+=1
            if wait>=patience: break
    model.load_state_dict(best_state)
    va_p,te_p=predict(model,va_dl),predict(model,te_dl)
    thr,_=pick_threshold_on_val(y[va],va_p)
    m={'model':'LSTM','matrix':label,**evaluate(y[te],te_p,threshold=thr)}
    sc=pd.concat([pd.DataFrame({'person_id':persons[va],'y_true':y[va],'proba':va_p,'split':'val'}),
                  pd.DataFrame({'person_id':persons[te],'y_true':y[te],'proba':te_p,'split':'test'})],ignore_index=True)
    sc.insert(0,'model','LSTM'); sc.insert(1,'matrix',label); sc['threshold']=thr
    sc.to_parquet(f'{SCORE_DIR}/LSTM_{label}.parquet',index=False)
    torch.save({'state_dict':best_state,'D_seq':D_seq,'d_static':d_static,'use_survey':use_survey,'threshold':thr,'label':label},
               f'{MODEL_DIR}/LSTM_{label}.pt')
    print(f"[LSTM {label:<22}] PR-AUC={m['PR_AUC']:.4f} ROC-AUC={m['ROC_AUC']:.4f} F1+={m['F1+']:.4f} (best_AP={best_ap:.4f}, thr={thr:.3f})")
    return m

In [ ]:
m_nosurv = run_lstm(use_survey=False)

In [ ]:
m_surv = run_lstm(use_survey=True)

12&24

In [ ]:
import os, gc, json, joblib
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

BASE='/home/jupyter/workspace/rw-migration-aou-rw-24b38658'
CLEAN_DIR=f'{BASE}/amia/clean_data'; feature_dir=f'{BASE}/amia/feature'; SEQ_DIR=f'{BASE}/amia/seq_data'
MODS={'condition':'condition_start_datetime','drug':'drug_exposure_start_datetime',
      'lab':'measurement_datetime','measurement':'measurement_datetime','observation':'observation_datetime'}

def prepare_seq(TF, chunksize=1_000_000):
    K = K_BY_TF[TF]                      
    print(f"\n===== prepare_seq TF={TF} K={K} =====")
    pos=pd.read_csv(f"{CLEAN_DIR}/clean_positive_{TF}.csv",usecols=['person_id','index_date','timeframe_start'])
    neg=pd.read_csv(f"{CLEAN_DIR}/clean_negative_anchor_{TF}.csv",usecols=['person_id','index_date','timeframe_start'])
    pos['IsPositive']=1; neg['IsPositive']=0
    anchors=pd.concat([pos,neg],ignore_index=True); assert anchors['person_id'].is_unique
    for c in ['index_date','timeframe_start']:
        anchors[c+'_i']=pd.to_numeric(pd.to_datetime(anchors[c],utc=True,errors='coerce').dt.strftime('%Y%m%d'),errors='coerce').astype('int32')
    bounds=anchors[['person_id','timeframe_start_i','index_date_i']].rename(columns={'timeframe_start_i':'ts','index_date_i':'ix'})
    persons=anchors['person_id'].to_numpy(); N=len(persons); y=anchors['IsPositive'].to_numpy().astype('int8')
    pid=anchors['person_id']; lab=anchors['IsPositive']
    ptr,pte,ytr,yte=train_test_split(pid,lab,test_size=0.20,stratify=lab,random_state=42)
    ptr,pva,ytr,yva=train_test_split(ptr,ytr,test_size=0.15,stratify=ytr,random_state=42)
    tr_ids,va_ids,te_ids=set(ptr),set(pva),set(pte)
    json.dump({'train':sorted(int(x) for x in tr_ids),'val':sorted(int(x) for x in va_ids),'test':sorted(int(x) for x in te_ids)},
              open(f'{feature_dir}/split_person_ids_{TF}.json','w'))
    split_arr=np.where(anchors['person_id'].isin(tr_ids),0,np.where(anchors['person_id'].isin(va_ids),1,2)).astype('int8')
    pid_to_row={int(p):i for i,p in enumerate(persons)}

    def load_pairs(fn,dt):
        p=f"{CLEAN_DIR}/{fn}.csv"
        if not os.path.exists(p): return pd.DataFrame(columns=['person_id','day'])
        out=[]
        for ch in pd.read_csv(p,usecols=['person_id',dt],dtype={dt:str},chunksize=chunksize):
            day=pd.to_numeric(ch[dt].str.slice(0,10).str.replace('-','',regex=False),errors='coerce')
            d=pd.DataFrame({'person_id':ch['person_id'].values,'day':day.values}).dropna(subset=['day'])
            d['day']=d['day'].astype('int32'); d=d.merge(bounds,on='person_id',how='inner')
            d=d.loc[(d['day']>=d['ts'])&(d['day']<d['ix']),['person_id','day']].drop_duplicates()
            if len(d): out.append(d)
            del ch,day,d
        gc.collect()
        return pd.concat(out,ignore_index=True).drop_duplicates() if out else pd.DataFrame(columns=['person_id','day'])
    ev=[]
    for mod,dt in MODS.items():
        ev+=[load_pairs(f"clean_{mod}_{TF}",dt),load_pairs(f"clean_negative_{mod}_{TF}",dt)]
    ev=pd.concat(ev,ignore_index=True).drop_duplicates().sort_values(['person_id','day'])
    day_mat=np.zeros((N,K),dtype='int32'); n_real=np.zeros(N,dtype='int16')
    for p,gg in ev.groupby('person_id',sort=False):
        r=pid_to_row.get(int(p),-1)
        if r<0: continue
        dd=gg['day'].to_numpy()
        if len(dd)>=K: day_mat[r]=dd[-K:]; n_real[r]=K
        else: m=len(dd); day_mat[r,:m]=dd; day_mat[r,m:]=dd[-1]; n_real[r]=m
    np.savez(f'{SEQ_DIR}/grid_{TF}.npz',person_id=persons.astype('int64'),day_mat=day_mat,n_real=n_real,y=y,split=split_arr)
    print(f"  grid N={N} ev={len(ev):,} n_real0={(n_real==0).sum()}"); del ev; gc.collect()

    row_s=pd.Series(np.arange(N,dtype='int32'),index=persons)
    long_grid=pd.DataFrame({'row':np.repeat(np.arange(N,dtype='int32'),K),'slot':np.tile(np.arange(K,dtype='int16'),N),'day':day_mat.reshape(-1)})
    long_grid=long_grid[long_grid['day']>0]

    def build_count(mod,dt):
        def lc(fn):
            p=f"{CLEAN_DIR}/{fn}.csv"
            if not os.path.exists(p): return pd.DataFrame(columns=['person_id','day','code','cnt'])
            out=[]
            for ch in pd.read_csv(p,usecols=['person_id','standard_concept_code',dt],dtype={dt:str},chunksize=chunksize):
                day=pd.to_numeric(ch[dt].str.slice(0,10).str.replace('-','',regex=False),errors='coerce')
                d=pd.DataFrame({'person_id':ch['person_id'].values,'day':day.values,'code':ch['standard_concept_code'].astype(str).values}).dropna(subset=['day'])
                d['day']=d['day'].astype('int32'); d=d.merge(bounds,on='person_id',how='inner')
                d=d.loc[(d['day']>=d['ts'])&(d['day']<d['ix']),['person_id','day','code']]
                if len(d): out.append(d.groupby(['person_id','day','code'],sort=False).size().reset_index(name='cnt'))
                del ch,day,d
            gc.collect()
            return pd.concat(out,ignore_index=True) if out else pd.DataFrame(columns=['person_id','day','code','cnt'])
        e=pd.concat([lc(f"clean_{mod}_{TF}"),lc(f"clean_negative_{mod}_{TF}")],ignore_index=True)
        e=e.groupby(['person_id','day','code'],sort=False)['cnt'].sum().reset_index()
        thr=0.005*len(tr_ids); prev=e[e['person_id'].isin(tr_ids)].groupby('code')['person_id'].nunique()
        vocab=sorted(prev[prev>=thr].index.tolist()); c2f={c:i for i,c in enumerate(vocab)}; D=len(vocab)
        e=e[e['code'].isin(set(vocab))].copy(); e['feat']=e['code'].map(c2f).astype('int32'); e['row']=e['person_id'].map(row_s).astype('int32')
        spx=long_grid.merge(e[['row','day','feat','cnt']],on=['row','day'],how='inner'); spx['val']=np.log1p(spx['cnt'].astype('float32')).astype('float32')
        np.savez(f'{SEQ_DIR}/seq_{mod}_{TF}.npz',row=spx['row'].to_numpy('int32'),slot=spx['slot'].to_numpy('int16'),feat=spx['feat'].to_numpy('int32'),val=spx['val'].to_numpy('float32'),N=np.int64(N),K=np.int64(K),D=np.int64(D))
        json.dump({'codes':vocab},open(f'{SEQ_DIR}/vocab_{mod}_{TF}.json','w')); print(f"  {mod:11s} D={D:>4} none zero={len(spx):>10,}"); del e,spx; gc.collect()
    for mod,dt in MODS.items(): build_count(mod,dt)

    def align(df):
        df=df.drop(columns=[c for c in df.columns if c.lower() in ('ispositive','label','is_positive')],errors='ignore')
        return df.set_index('person_id').reindex(persons)
    demo=pd.read_csv(f'{CLEAN_DIR}/demographic.csv'); surv = pd.read_parquet(f'{BASE}/amia/new_survey/survey_features_{TF}.parquet')
    labf=pd.read_parquet(f'{feature_dir}/lab_features_{TF}.parquet'); measf=pd.read_parquet(f'{feature_dir}/measurement_features_{TF}.parquet')
    labnum=labf[['person_id']+[c for c in labf.columns if c.startswith(('lab_mean_','lab_last_'))]]
    measnum=measf[['person_id']+[c for c in measf.columns if c.endswith(('_mean','_last'))]]
    base=pd.concat([align(demo),align(labnum),align(measnum)],axis=1); full=pd.concat([base,align(surv)],axis=1)
    tm=(split_arr==0)
    def fs(mat):
        X=mat.to_numpy('float32'); imp=SimpleImputer(strategy='median'); sc=StandardScaler(); imp.fit(X[tm]); Xi=imp.transform(X); sc.fit(Xi[tm]); return sc.transform(Xi).astype('float32'),imp,sc
    Xb,ib,sb=fs(base); Xf,ifu,sf=fs(full)
    np.savez(f'{SEQ_DIR}/static_{TF}.npz',static_nosurv=Xb,static_surv=Xf)
    joblib.dump({'nosurv':{'cols':base.columns.tolist(),'imputer':ib,'scaler':sb},'surv':{'cols':full.columns.tolist(),'imputer':ifu,'scaler':sf}},f'{SEQ_DIR}/static_prep_{TF}.joblib')
    print(f"  static w/o={base.shape[1]} w/={full.shape[1]}"); del labf,measf; gc.collect()

K_BY_TF = {6: 20, 12: 30, 24: 50}
for TF in [12, 24]:
    prepare_seq(TF)

In [ ]:
import numpy as np, pandas as pd, gc, os, torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (roc_auc_score, average_precision_score, precision_score, recall_score, f1_score)
torch.set_num_threads(4)
BASE='/home/jupyter/workspace/rw-migration-aou-rw-24b38658'
SEQ_DIR=f'{BASE}/amia/seq_data'; MODEL_DIR=f'{BASE}/amia/new_survey/models'; SCORE_DIR=f'{BASE}/amia/new_survey/risk_scores';
K_BY_TF = {6: 20, 12: 30, 24: 50}
K = K_BY_TF[TF]      
def pick_threshold_on_val(y_val, proba_val, grid=np.linspace(0.05,0.95,181)):
    y_val=np.asarray(y_val); bt,bf=0.5,-1.0
    for t in grid:
        f=f1_score(y_val,(proba_val>=t).astype(int),pos_label=1,zero_division=0)
        if f>bf: bf,bt=f,float(t)
    return bt,bf
def evaluate(y_true, proba, threshold=None):
    y_true=np.asarray(y_true)
    if threshold is None: threshold,_=pick_threshold_on_val(y_true,proba)
    yp=(proba>=threshold).astype(int)
    return {'Macro_AUC':roc_auc_score(y_true,proba),'Precision+':precision_score(y_true,yp,pos_label=1,zero_division=0),
            'Recall+':recall_score(y_true,yp,pos_label=1,zero_division=0),'F1+':f1_score(y_true,yp,pos_label=1,zero_division=0),
            'Macro_F1':f1_score(y_true,yp,average='macro'),'ROC_AUC':roc_auc_score(y_true,proba),
            'PR_AUC':average_precision_score(y_true,proba),'threshold':threshold}

class DS(Dataset):
    def __init__(s,rows,static,rp,sl,ft,vl,y,K,D,nr): s.rows,s.st,s.rp,s.sl,s.ft,s.vl,s.y,s.K,s.D,s.nr=rows,static,rp,sl,ft,vl,y,K,D,nr
    def __len__(s): return len(s.rows)
    def __getitem__(s,i):
        r=s.rows[i]; a,b=s.rp[r],s.rp[r+1]; x=np.zeros((s.K,s.D),dtype='float32'); x[s.sl[a:b],s.ft[a:b]]=s.vl[a:b]
        L=max(int(s.nr[r]),1)                                       
        return x,s.st[r],s.y[r],np.int64(L)

class TwoTower(nn.Module):
    def __init__(s,d_seq,d_static,hs=128,ht=64,p=0.3):
        super().__init__(); s.lstm=nn.LSTM(d_seq,hs,batch_first=True)
        s.stat=nn.Sequential(nn.Linear(d_static,ht),nn.ReLU(),nn.Dropout(p))
        s.head=nn.Sequential(nn.Linear(hs+ht,64),nn.ReLU(),nn.Dropout(p),nn.Linear(64,1))
    def forward(s,xs,xt,lens):
        packed=nn.utils.rnn.pack_padded_sequence(xs,lens.cpu(),batch_first=True,enforce_sorted=False)
        _,(h,_)=s.lstm(packed)                                       
        return s.head(torch.cat([h[-1],s.stat(xt)],1)).squeeze(1)

def run_lstm_tf(TF, use_survey, max_epochs=30, patience=5, bs=512):
    K = K_BY_TF[TF]
    g=np.load(f'{SEQ_DIR}/grid_{TF}.npz'); persons=g['person_id']; y=g['y'].astype('float32'); split=g['split']; nr=g['n_real'].astype('int64'); N=len(persons)
    R,Sl,F,V,off=[],[],[],[],0
    for mod in ['condition','drug','observation','lab','measurement']:
        z=np.load(f'{SEQ_DIR}/seq_{mod}_{TF}.npz'); D=int(z['D'])
        R.append(z['row']); Sl.append(z['slot']); F.append(z['feat'].astype('int32')+off); V.append(z['val']); off+=D
    D_seq=off; row=np.concatenate(R); slot=np.concatenate(Sl).astype('int16'); feat=np.concatenate(F); val=np.concatenate(V)
    o=np.argsort(row,kind='stable'); row,slot,feat,val=row[o],slot[o],feat[o],val[o]; rp=np.searchsorted(row,np.arange(N+1)).astype('int64')
    del R,Sl,F,V,o; gc.collect()
    S=np.load(f'{SEQ_DIR}/static_{TF}.npz'); static=S['static_surv'] if use_survey else S['static_nosurv']; d_static=static.shape[1]
    ver='ehrdemo_survnobasic' if use_survey else 'ehrdemo'; label=f'{ver}_{TF}m'
    tr=np.where(split==0)[0]; va=np.where(split==1)[0]; te=np.where(split==2)[0]
    mk=lambda rows,sh: DataLoader(DS(rows,static,rp,slot,feat,val,y,K,D_seq,nr),batch_size=(bs if sh else 1024),shuffle=sh,num_workers=0)
    tr_dl,va_dl,te_dl=mk(tr,True),mk(va,False),mk(te,False)
    def pred(m,dl):
        m.eval(); ps=[]
        with torch.no_grad():
            for xs,xt,_,ln in dl: ps.append(torch.sigmoid(m(xs,xt,ln)).numpy())
        return np.concatenate(ps)
    torch.manual_seed(42); model=TwoTower(D_seq,d_static); opt=torch.optim.Adam(model.parameters(),lr=1e-3,weight_decay=1e-5); lf=nn.BCEWithLogitsLoss()
    best,bst,wait=-1,None,0
    for ep in range(max_epochs):
        model.train()
        for xs,xt,yb,ln in tr_dl: opt.zero_grad(); lf(model(xs,xt,ln),yb).backward(); opt.step()
        ap=average_precision_score(y[va],pred(model,va_dl)); print(f"  [{label}] ep{ep+1:>2} val_AP={ap:.4f}")
        if ap>best: best,bst,wait=ap,{k:v.clone() for k,v in model.state_dict().items()},0
        else:
            wait+=1
            if wait>=patience: break
    model.load_state_dict(bst); vp,tp=pred(model,va_dl),pred(model,te_dl); thr,_=pick_threshold_on_val(y[va],vp)
    m={'model':'LSTM','matrix':label,**evaluate(y[te],tp,threshold=thr)}
    sc=pd.concat([pd.DataFrame({'person_id':persons[va],'y_true':y[va],'proba':vp,'split':'val'}),
                  pd.DataFrame({'person_id':persons[te],'y_true':y[te],'proba':tp,'split':'test'})],ignore_index=True)
    sc.insert(0,'model','LSTM'); sc.insert(1,'matrix',label); sc['threshold']=thr; sc.to_parquet(f'{SCORE_DIR}/LSTM_{label}.parquet',index=False)
    torch.save({'state_dict':bst,'D_seq':D_seq,'d_static':d_static,'use_survey':use_survey,
            'threshold':thr,'label':label,'K':K,'TF':TF},
           f'{MODEL_DIR}/LSTM_{label}.pt')
    print(f"[LSTM {label:<22}] PR-AUC={m['PR_AUC']:.4f} ROC-AUC={m['ROC_AUC']:.4f} F1+={m['F1+']:.4f} (best_AP={best:.4f})")
    return m

for TF in [12,24]:
    run_lstm_tf(TF, use_survey=False)
    run_lstm_tf(TF, use_survey=True)

In [ ]:
import glob, numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, precision_score, recall_score
SCORE_DIR=f'{BASE}/amia/risk_scores'
rows=[]
for p in glob.glob(f'{SCORE_DIR}/*.parquet'):
    d=pd.read_parquet(p); te=d[d['split']=='test']; thr=te['threshold'].iloc[0]; yp=(te['proba']>=thr).astype(int)
    rows.append({'model':te['model'].iloc[0],'matrix':te['matrix'].iloc[0],
                 'ROC_AUC':roc_auc_score(te['y_true'],te['proba']),'PR_AUC':average_precision_score(te['y_true'],te['proba']),
                 'F1+':f1_score(te['y_true'],yp,zero_division=0),'Precision+':precision_score(te['y_true'],yp,zero_division=0),
                 'Recall+':recall_score(te['y_true'],yp,zero_division=0)})
master=pd.DataFrame(rows)
master['tf']=master['matrix'].str.extract(r'_(\d+)m$').astype(int)
master['featureset']=np.where(master['matrix'].str.contains('survnobasic'),'EHR+Survey','EHR')
master=master.sort_values(['model','tf','featureset']).reset_index(drop=True)
master.to_csv(f'{BASE}/amia/all_results.csv',index=False)
print(master.to_string(index=False))
piv=master.pivot_table(index=['model','tf'],columns='featureset',values='PR_AUC')
piv['Δ_PR']=piv['EHR+Survey']-piv['EHR']; print("\n", piv.round(4))

RNN & Transformer

In [ ]:
import os, time
for TF in [6, 12, 24]:
    p = f'{SEQ_DIR}/static_{TF}.npz'
    if not os.path.exists(p):
        print(f"{TF}m: file not exist"); continue
    z = np.load(p)
    print(f"{TF}m: nosurv={z['static_nosurv'].shape} surv={z['static_surv'].shape} "
          f"| mtime={time.strftime('%m-%d %H:%M', time.localtime(os.path.getmtime(p)))}")

In [ ]:
import numpy as np, pandas as pd, gc, os, torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (roc_auc_score, average_precision_score, precision_score, recall_score, f1_score)
torch.set_num_threads(8)
BASE='/home/jupyter/workspace/rw-migration-aou-rw-24b38658'
SEQ_DIR=f'{BASE}/amia/seq_data'; MODEL_DIR=f'{BASE}/amia/new_survey/models'; SCORE_DIR=f'{BASE}/amia/new_survey/risk_scores'; 
device='cuda' if torch.cuda.is_available() else 'cpu'; print("device =", device)

def pick_threshold_on_val(y_val, proba_val, grid=np.linspace(0.05,0.95,181)):
    y_val=np.asarray(y_val); bt,bf=0.5,-1.0
    for t in grid:
        f=f1_score(y_val,(proba_val>=t).astype(int),pos_label=1,zero_division=0)
        if f>bf: bf,bt=f,float(t)
    return bt,bf
def evaluate(y_true, proba, threshold=None):
    y_true=np.asarray(y_true)
    if threshold is None: threshold,_=pick_threshold_on_val(y_true,proba)
    yp=(proba>=threshold).astype(int)
    return {'Macro_AUC':roc_auc_score(y_true,proba),'Precision+':precision_score(y_true,yp,pos_label=1,zero_division=0),
            'Recall+':recall_score(y_true,yp,pos_label=1,zero_division=0),'F1+':f1_score(y_true,yp,pos_label=1,zero_division=0),
            'Macro_F1':f1_score(y_true,yp,average='macro'),'ROC_AUC':roc_auc_score(y_true,proba),
            'PR_AUC':average_precision_score(y_true,proba),'threshold':threshold}

class DS(Dataset):
    def __init__(s,rows,static,rp,sl,ft,vl,y,K,D,nr): s.rows,s.st,s.rp,s.sl,s.ft,s.vl,s.y,s.K,s.D,s.nr=rows,static,rp,sl,ft,vl,y,K,D,nr
    def __len__(s): return len(s.rows)
    def __getitem__(s,i):
        r=int(s.rows[i]); a,b=int(s.rp[r]),int(s.rp[r+1]); x=np.zeros((s.K,s.D),dtype='float32'); x[s.sl[a:b],s.ft[a:b]]=s.vl[a:b]
        L=max(int(s.nr[r]),1)                                      
        return x, s.st[r], np.float32(s.y[r]), np.int64(L)

class SeqTower(nn.Module):
    def __init__(s,kind,d_seq,d_static,K,hs=128,ht=64,p=0.3,d_model=128,nhead=4,nlayers=2):
        super().__init__(); s.kind=kind
        if kind in ('LSTM','GRU'):
            s.enc=(nn.LSTM if kind=='LSTM' else nn.GRU)(d_seq,hs,batch_first=True); s.out=hs
        else:
            s.proj=nn.Linear(d_seq,d_model); s.pos=nn.Parameter(torch.randn(K,d_model)*0.02)
            s.enc=nn.TransformerEncoder(nn.TransformerEncoderLayer(d_model,nhead,256,p,batch_first=True),nlayers); s.out=d_model
        s.stat=nn.Sequential(nn.Linear(d_static,ht),nn.ReLU(),nn.Dropout(p))
        s.head=nn.Sequential(nn.Linear(s.out+ht,64),nn.ReLU(),nn.Dropout(p),nn.Linear(64,1))
    def forward(s,xs,xt,lens):
        lens=lens.to(torch.long)
        if s.kind in ('LSTM','GRU'):
            packed=nn.utils.rnn.pack_padded_sequence(xs,lens.cpu(),batch_first=True,enforce_sorted=False)
            if s.kind=='LSTM': _,(h,_)=s.enc(packed)
            else:              _,h=s.enc(packed)
            hseq=h[-1]
        else:
            kpm=torch.arange(xs.size(1),device=xs.device).unsqueeze(0) >= lens.to(xs.device).unsqueeze(1)  
            z=s.enc(s.proj(xs)+s.pos, src_key_padding_mask=kpm)    
            w=(~kpm).float().unsqueeze(-1)                         
            hseq=(z*w).sum(1)/w.sum(1).clamp(min=1e-6)
        return s.head(torch.cat([hseq,s.stat(xt)],1)).squeeze(1)

def run_seq_tf(kind, model_name, prefix, TF, use_survey, max_epochs=30, patience=5, bs=512, quiet=False):
    K = K_BY_TF[TF]     
    g=np.load(f'{SEQ_DIR}/grid_{TF}.npz'); persons=g['person_id']; y=g['y'].astype('float32'); split=g['split']; nr=g['n_real'].astype('int64'); N=len(persons)
    R,Sl,F,V,off=[],[],[],[],0
    for mod in ['condition','drug','observation','lab','measurement']:
        z=np.load(f'{SEQ_DIR}/seq_{mod}_{TF}.npz'); D=int(z['D'])
        R.append(z['row']); Sl.append(z['slot']); F.append(z['feat'].astype('int32')+off); V.append(z['val']); off+=D
    D_seq=off; row=np.concatenate(R); slot=np.concatenate(Sl).astype('int64'); feat=np.concatenate(F).astype('int64'); val=np.concatenate(V)
    o=np.argsort(row,kind='stable'); row,slot,feat,val=row[o],slot[o],feat[o],val[o]; rp=np.searchsorted(row,np.arange(N+1)).astype('int64'); del R,Sl,F,V,o; gc.collect()
    Sm=np.load(f'{SEQ_DIR}/static_{TF}.npz'); static=Sm['static_surv'] if use_survey else Sm['static_nosurv']; d_static=static.shape[1]
    ver='ehrdemo_survnobasic' if use_survey else 'ehrdemo'; label=f'{ver}_{TF}m'
    tr=np.where(split==0)[0]; va=np.where(split==1)[0]; te=np.where(split==2)[0]
    mk=lambda rows,sh: DataLoader(DS(rows,static,rp,slot,feat,val,y,K,D_seq,nr),batch_size=(bs if sh else 2048),shuffle=sh,num_workers=0,pin_memory=(device=='cuda'))
    tr_dl,va_dl,te_dl=mk(tr,True),mk(va,False),mk(te,False)
    def pred(m,dl):
        m.eval(); ps=[]
        with torch.no_grad():
            for xs,xt,_,ln in dl: ps.append(torch.sigmoid(m(xs.to(device),xt.to(device),ln)).cpu().numpy())
        return np.concatenate(ps)
    torch.manual_seed(42); model=SeqTower(kind,D_seq,d_static,K).to(device)   
    opt=torch.optim.Adam(model.parameters(),lr=1e-3,weight_decay=1e-5); lf=nn.BCEWithLogitsLoss()
    best,bst,wait=-1.0,None,0
    for ep in range(max_epochs):
        model.train()
        for xs,xt,yb,ln in tr_dl:
            xs,xt,yb=xs.to(device),xt.to(device),yb.to(device)
            opt.zero_grad(); lf(model(xs,xt,ln),yb).backward(); opt.step()
        ap=average_precision_score(y[va],pred(model,va_dl))
        if not quiet: print(f"  ep{ep+1:>2} val_AP={ap:.4f}")
        if ap>best: best,bst,wait=ap,{k:v.detach().cpu().clone() for k,v in model.state_dict().items()},0
        else:
            wait+=1
            if wait>=patience: break
    model.load_state_dict(bst); vp,tp=pred(model,va_dl),pred(model,te_dl); thr,_=pick_threshold_on_val(y[va],vp)
    m={'model':model_name,'matrix':label,**evaluate(y[te],tp,threshold=thr)}
    sc=pd.concat([pd.DataFrame({'person_id':persons[va],'y_true':y[va],'proba':vp,'split':'val'}),
                  pd.DataFrame({'person_id':persons[te],'y_true':y[te],'proba':tp,'split':'test'})],ignore_index=True)
    sc.insert(0,'model',model_name); sc.insert(1,'matrix',label); sc['threshold']=thr
    sc.to_parquet(f'{SCORE_DIR}/{prefix}_{label}.parquet',index=False)
    torch.save({'state_dict':bst,'kind':kind,'D_seq':D_seq,'d_static':d_static,'use_survey':use_survey,
                'threshold':thr,'label':label,'K':K,'TF':TF},
               f'{MODEL_DIR}/{prefix}_{label}.pt')
    print(f"[{model_name} {label:<22}] PR-AUC={m['PR_AUC']:.4f} ROC-AUC={m['ROC_AUC']:.4f} F1+={m['F1+']:.4f} (best_AP={best:.4f})")
    return m

for TF in [6]:
    for us in [False, True]:
        run_seq_tf('GRU', 'RNN', 'RNN', TF, us)

In [ ]:
for TF in [12]:
    for us in [False, True]:
        run_seq_tf('GRU', 'RNN', 'RNN', TF, us)

In [ ]:
for TF in [24]:
    for us in [False, True]:
        run_seq_tf('GRU', 'RNN', 'RNN', TF, us)

In [ ]:
import glob, os
SCORE_DIR=f'{BASE}/amia/new_survey/risk_scores'
for pref in ['LSTM','RNN','TRANS']:
    print(pref, sorted(os.path.basename(x).replace(f'{pref}_','').replace('.parquet','')
                       for x in glob.glob(f'{SCORE_DIR}/{pref}_*.parquet')))

In [ ]:
import numpy as np, pandas as pd, gc, os, torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (roc_auc_score, average_precision_score, precision_score, recall_score, f1_score)
torch.set_num_threads(8)
BASE='/home/jupyter/workspace/rw-migration-aou-rw-24b38658'
SEQ_DIR=f'{BASE}/amia/seq_data'; MODEL_DIR=f'{BASE}/amia/new_survey/models'; SCORE_DIR=f'{BASE}/amia/new_survey/risk_scores'; 
device='cuda' if torch.cuda.is_available() else 'cpu'; print("device =", device)

def pick_threshold_on_val(y_val, proba_val, grid=np.linspace(0.05,0.95,181)):
    y_val=np.asarray(y_val); bt,bf=0.5,-1.0
    for t in grid:
        f=f1_score(y_val,(proba_val>=t).astype(int),pos_label=1,zero_division=0)
        if f>bf: bf,bt=f,float(t)
    return bt,bf
def evaluate(y_true, proba, threshold=None):
    y_true=np.asarray(y_true)
    if threshold is None: threshold,_=pick_threshold_on_val(y_true,proba)
    yp=(proba>=threshold).astype(int)
    return {'Macro_AUC':roc_auc_score(y_true,proba),'Precision+':precision_score(y_true,yp,pos_label=1,zero_division=0),
            'Recall+':recall_score(y_true,yp,pos_label=1,zero_division=0),'F1+':f1_score(y_true,yp,pos_label=1,zero_division=0),
            'Macro_F1':f1_score(y_true,yp,average='macro'),'ROC_AUC':roc_auc_score(y_true,proba),
            'PR_AUC':average_precision_score(y_true,proba),'threshold':threshold}

class DS(Dataset):
    def __init__(s,rows,static,rp,sl,ft,vl,y,K,D,nr): s.rows,s.st,s.rp,s.sl,s.ft,s.vl,s.y,s.K,s.D,s.nr=rows,static,rp,sl,ft,vl,y,K,D,nr
    def __len__(s): return len(s.rows)
    def __getitem__(s,i):
        r=int(s.rows[i]); a,b=int(s.rp[r]),int(s.rp[r+1]); x=np.zeros((s.K,s.D),dtype='float32'); x[s.sl[a:b],s.ft[a:b]]=s.vl[a:b]
        L=max(int(s.nr[r]),1)                                       
        return x, s.st[r], np.float32(s.y[r]), np.int64(L)

class SeqTower(nn.Module):
    def __init__(s,kind,d_seq,d_static,K,hs=128,ht=64,p=0.3,d_model=128,nhead=4,nlayers=2):
        super().__init__(); s.kind=kind
        if kind in ('LSTM','GRU'):
            s.enc=(nn.LSTM if kind=='LSTM' else nn.GRU)(d_seq,hs,batch_first=True); s.out=hs
        else:
            s.proj=nn.Linear(d_seq,d_model); s.pos=nn.Parameter(torch.randn(K,d_model)*0.02)
            s.enc=nn.TransformerEncoder(nn.TransformerEncoderLayer(d_model,nhead,256,p,batch_first=True),nlayers); s.out=d_model
        s.stat=nn.Sequential(nn.Linear(d_static,ht),nn.ReLU(),nn.Dropout(p))
        s.head=nn.Sequential(nn.Linear(s.out+ht,64),nn.ReLU(),nn.Dropout(p),nn.Linear(64,1))
    def forward(s,xs,xt,lens):
        lens=lens.to(torch.long)
        if s.kind in ('LSTM','GRU'):
            packed=nn.utils.rnn.pack_padded_sequence(xs,lens.cpu(),batch_first=True,enforce_sorted=False)
            if s.kind=='LSTM': _,(h,_)=s.enc(packed)
            else:              _,h=s.enc(packed)
            hseq=h[-1]
        else:
            kpm=torch.arange(xs.size(1),device=xs.device).unsqueeze(0) >= lens.to(xs.device).unsqueeze(1)  
            z=s.enc(s.proj(xs)+s.pos, src_key_padding_mask=kpm)    
            w=(~kpm).float().unsqueeze(-1)                         
            hseq=(z*w).sum(1)/w.sum(1).clamp(min=1e-6)
        return s.head(torch.cat([hseq,s.stat(xt)],1)).squeeze(1)

def run_seq_tf(kind, model_name, prefix, TF, use_survey, max_epochs=30, patience=5, bs=512, quiet=False):
    K = K_BY_TF[TF]     
    g=np.load(f'{SEQ_DIR}/grid_{TF}.npz'); persons=g['person_id']; y=g['y'].astype('float32'); split=g['split']; nr=g['n_real'].astype('int64'); N=len(persons)
    R,Sl,F,V,off=[],[],[],[],0
    for mod in ['condition','drug','observation','lab','measurement']:
        z=np.load(f'{SEQ_DIR}/seq_{mod}_{TF}.npz'); D=int(z['D'])
        R.append(z['row']); Sl.append(z['slot']); F.append(z['feat'].astype('int32')+off); V.append(z['val']); off+=D
    D_seq=off; row=np.concatenate(R); slot=np.concatenate(Sl).astype('int64'); feat=np.concatenate(F).astype('int64'); val=np.concatenate(V)
    o=np.argsort(row,kind='stable'); row,slot,feat,val=row[o],slot[o],feat[o],val[o]; rp=np.searchsorted(row,np.arange(N+1)).astype('int64'); del R,Sl,F,V,o; gc.collect()
    Sm=np.load(f'{SEQ_DIR}/static_{TF}.npz'); static=Sm['static_surv'] if use_survey else Sm['static_nosurv']; d_static=static.shape[1]
    ver='ehrdemo_survnobasic' if use_survey else 'ehrdemo'; label=f'{ver}_{TF}m'
    tr=np.where(split==0)[0]; va=np.where(split==1)[0]; te=np.where(split==2)[0]
    mk=lambda rows,sh: DataLoader(DS(rows,static,rp,slot,feat,val,y,K,D_seq,nr),batch_size=(bs if sh else 2048),shuffle=sh,num_workers=0,pin_memory=(device=='cuda'))
    tr_dl,va_dl,te_dl=mk(tr,True),mk(va,False),mk(te,False)
    def pred(m,dl):
        m.eval(); ps=[]
        with torch.no_grad():
            for xs,xt,_,ln in dl: ps.append(torch.sigmoid(m(xs.to(device),xt.to(device),ln)).cpu().numpy())
        return np.concatenate(ps)
    torch.manual_seed(42); model=SeqTower(kind,D_seq,d_static,K).to(device)   
    opt=torch.optim.Adam(model.parameters(),lr=1e-3,weight_decay=1e-5); lf=nn.BCEWithLogitsLoss()
    best,bst,wait=-1.0,None,0
    for ep in range(max_epochs):
        model.train()
        for xs,xt,yb,ln in tr_dl:
            xs,xt,yb=xs.to(device),xt.to(device),yb.to(device)
            opt.zero_grad(); lf(model(xs,xt,ln),yb).backward(); opt.step()
        ap=average_precision_score(y[va],pred(model,va_dl))
        if not quiet: print(f"  ep{ep+1:>2} val_AP={ap:.4f}")
        if ap>best: best,bst,wait=ap,{k:v.detach().cpu().clone() for k,v in model.state_dict().items()},0
        else:
            wait+=1
            if wait>=patience: break
    model.load_state_dict(bst); vp,tp=pred(model,va_dl),pred(model,te_dl); thr,_=pick_threshold_on_val(y[va],vp)
    m={'model':model_name,'matrix':label,**evaluate(y[te],tp,threshold=thr)}
    sc=pd.concat([pd.DataFrame({'person_id':persons[va],'y_true':y[va],'proba':vp,'split':'val'}),
                  pd.DataFrame({'person_id':persons[te],'y_true':y[te],'proba':tp,'split':'test'})],ignore_index=True)
    sc.insert(0,'model',model_name); sc.insert(1,'matrix',label); sc['threshold']=thr
    sc.to_parquet(f'{SCORE_DIR}/{prefix}_{label}.parquet',index=False)
    torch.save({'state_dict':bst,'kind':kind,'D_seq':D_seq,'d_static':d_static,'use_survey':use_survey,
                'threshold':thr,'label':label,'K':K,'TF':TF},
               f'{MODEL_DIR}/{prefix}_{label}.pt')
    print(f"[{model_name} {label:<22}] PR-AUC={m['PR_AUC']:.4f} ROC-AUC={m['ROC_AUC']:.4f} F1+={m['F1+']:.4f} (best_AP={best:.4f})")
    return m

In [ ]:
run_seq_tf('Transformer','Transformer','TRANS',6,False)

In [ ]:
run_seq_tf('Transformer','Transformer','TRANS',6,True)

In [ ]:
run_seq_tf('Transformer','Transformer','TRANS',12,False)

In [ ]:
run_seq_tf('Transformer','Transformer','TRANS',12,True)

In [ ]:
run_seq_tf('Transformer','Transformer','TRANS',24,False)

In [ ]:
run_seq_tf('Transformer','Transformer','TRANS',24,True)

In [ ]:
import glob, os
for pref in ['LSTM','RNN','TRANS']:
    print(pref, sorted(os.path.basename(x).replace(f'{pref}_','').replace('.parquet','') for x in glob.glob(f'{SCORE_DIR}/{pref}_*.parquet')))

import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, precision_score, recall_score
rows=[]
for p in glob.glob(f'{SCORE_DIR}/*.parquet'):
    d=pd.read_parquet(p); te=d[d['split']=='test']; thr=te['threshold'].iloc[0]; yp=(te['proba']>=thr).astype(int)
    rows.append({'model':te['model'].iloc[0],'matrix':te['matrix'].iloc[0],
                 'ROC_AUC':roc_auc_score(te['y_true'],te['proba']),'PR_AUC':average_precision_score(te['y_true'],te['proba']),
                 'F1+':f1_score(te['y_true'],yp,zero_division=0),'Precision+':precision_score(te['y_true'],yp,zero_division=0),
                 'Recall+':recall_score(te['y_true'],yp,zero_division=0)})
master=pd.DataFrame(rows)
master['tf']=master['matrix'].str.extract(r'_(\d+)m$').astype(int)
master['featureset']=np.where(master['matrix'].str.contains('survnobasic'),'EHR+Survey','EHR')
master=master.sort_values(['model','tf','featureset']).reset_index(drop=True)
master.to_csv(f'{BASE}/amia/all_results.csv',index=False)
print(master.to_string(index=False))

In [ ]:
import pandas as pd, numpy as np
BASE='/home/jupyter/workspace/rw-migration-aou-rw-24b38658'
m = pd.read_csv(f'{BASE}/amia/all_results.csv')

def gain(metric):
    p = m.pivot_table(index=['model','tf'], columns='featureset', values=metric)
    p = p.rename(columns={'EHR':f'{metric}_EHR','EHR+Survey':f'{metric}_Surv'})
    p[f'{metric}_Δ'] = p[f'{metric}_Surv'] - p[f'{metric}_EHR']
    return p

tbl = gain('PR_AUC').join(gain('ROC_AUC')).reset_index()

order = ['LogisticReg','RandomForest','XGBoost','LightGBM','MLP','RNN','LSTM','Transformer']
tbl['model'] = pd.Categorical(tbl['model'], order)
tbl = tbl.sort_values(['model','tf']).reset_index(drop=True)
tbl = tbl[['model','tf','PR_AUC_EHR','PR_AUC_Surv','PR_AUC_Δ','ROC_AUC_EHR','ROC_AUC_Surv','ROC_AUC_Δ']]
for c in tbl.columns:
    if c not in ('model','tf'): tbl[c] = tbl[c].round(4)

tbl.to_csv(f'{BASE}/amia/survey_gain_table.csv', index=False)
print(tbl.to_string(index=False))

print("\neach timeframe average survey increment(PR-AUC):")
avg = m.pivot_table(index='tf', columns='featureset', values='PR_AUC')
avg['Δ_mean'] = avg['EHR+Survey'] - avg['EHR']
print(avg.round(4))